# Validation 기반 모델 비교 및 임계값 탐색

기본 모델과 클래스 가중치 적용 DenseNet121 U-ignore 모델을 비교하고, 가중치 적용 모델의 소견별 임계값 후보를 확인합니다.

- **위에서부터 순서대로 실행**하세요.
- 노트북을 프로젝트 루트 또는 `evaluation/`에 두세요. 아래 `PROJECT_DIR`을 직접 지정할 수도 있습니다.
- 기존 전처리 모듈, 데이터 CSV, 이미지 및 두 모델 체크포인트가 필요합니다.
- Validation만 추론하며 학습 및 Test 평가는 수행하지 않습니다.
- 원본 코드의 1,077장 검증과 평가 계산을 유지했습니다. F1 최대 임계값은 최종 확정값이 아닌 후보입니다.

## 1. 라이브러리 및 프로젝트 경로

In [ ]:
from __future__ import annotations

import csv
import json
from datetime import datetime
from pathlib import Path
import sys

import numpy as np
import torch
from sklearn.metrics import average_precision_score, roc_auc_score
from torchvision.models import densenet121
from torch import nn

# 프로젝트 루트 경로를 직접 입력할 수 있습니다.
PROJECT_DIR = None  # 예: Path("/home/team30103/chest-xray-multilabel")
CURRENT_DIR = Path.cwd().resolve()
ROOT = Path(PROJECT_DIR).expanduser().resolve() if PROJECT_DIR else (CURRENT_DIR.parent if CURRENT_DIR.name == "evaluation" else CURRENT_DIR)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cxr_preprocessing_uignore import IGNORE_INDEX, TARGET_LABELS, build_dataloaders
DEFAULT_BASELINE = ROOT / "runs/uignore_full/train_20261019_112210_839827/best_model.pth"
DEFAULT_WEIGHTED = ROOT / "runs/uignore_weighted_full/train_20261019_165121_403875/best_model.pth"
DEFAULT_IMAGE_ROOT = Path('/home/team30103/physionet_data')



import pandas as pd
from IPython.display import display
print("프로젝트:", ROOT)

## 2. 모델 및 평가 함수
소견별 불확실 라벨은 지표 계산에서 제외합니다. 그룹 분석은 세 소견 모두 확정 라벨인 영상에서 세 예측이 모두 맞는 비율을 계산합니다.

In [ ]:
class DenseNetUIgnore(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = densenet121(weights=None)  # no download; load the checkpoint
        self.backbone.classifier = nn.Linear(self.backbone.classifier.in_features, 6)

    def forward(self, images):
        return self.backbone(images).reshape(-1, 3, 2)


def load_model(checkpoint_path, device):
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"체크포인트 없음: {checkpoint_path}")
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    state = ckpt.get('model_state_dict', ckpt)
    if not isinstance(state, dict):
        raise ValueError('체크포인트에서 state_dict를 찾지 못했습니다.')
    if 'target_labels' in ckpt and tuple(ckpt['target_labels']) != tuple(TARGET_LABELS):
        raise ValueError(f"소견 순서 불일치: {ckpt['target_labels']}")
    if ckpt.get('label_strategy', 'u_ignore') != 'u_ignore':
        raise ValueError('U-ignore 체크포인트가 아닙니다.')
    model = DenseNetUIgnore()
    model.load_state_dict(state, strict=True)
    return model.to(device).eval(), ckpt


@torch.inference_mode()
def predict_validation(model, loader, device):
    true_list, prob_list = [], []
    for images, labels in loader:
        if labels.ndim != 2 or labels.shape[1] != 3:
            raise ValueError(f'라벨 차원 불일치: {labels.shape}')
        logits = model(images.to(device, non_blocking=True))
        pos = logits.softmax(-1)[:, :, 1].cpu().numpy()
        true_list.append(labels.numpy())
        prob_list.append(pos)
    y = np.concatenate(true_list, axis=0)
    p = np.concatenate(prob_list, axis=0)
    if y.shape != (1077, 3):
        raise ValueError(f'Validation 크기 예상 1077x3, 실제 {y.shape}')
    if not np.isin(y, [0, 1, IGNORE_INDEX]).all():
        raise ValueError('예상하지 못한 라벨 값')
    if not np.isfinite(p).all():
        raise ValueError('확률에 NaN 또는 Inf가 있습니다.')
    return y, p


def confusion_and_metrics(true, probabilities, threshold):
    valid = true != IGNORE_INDEX
    y = true[valid]
    p = probabilities[valid]
    pred = (p >= threshold).astype(int)
    tp = int(((y == 1) & (pred == 1)).sum())
    tn = int(((y == 0) & (pred == 0)).sum())
    fp = int(((y == 0) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    specificity = tn / (tn + fp) if tn + fp else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return dict(threshold=round(float(threshold), 2), n_valid=int(valid.sum()), positive=int((y==1).sum()),
                negative=int((y==0).sum()), TP=tp, TN=tn, FP=fp, FN=fn,
                precision=precision, recall=recall, specificity=specificity, positive_f1=f1)


def ranking_metrics(y, p):
    out=[]
    for k, finding in enumerate(TARGET_LABELS):
        mask=y[:,k] != IGNORE_INDEX
        labels=y[mask,k]
        scores=p[mask,k]
        if len(np.unique(labels)) == 2:
            auroc=float(roc_auc_score(labels,scores))
            ap=float(average_precision_score(labels,scores))
        else:
            auroc=ap=float('nan')
        out.append(dict(finding=finding, AUROC=auroc, AP=ap))
    return out


def group_accuracy(y, p, thresholds):
    # Complete-label cases ONLY; exact match across all 3 findings.
    complete=(y != IGNORE_INDEX).all(axis=1)
    actual=y[complete]
    predicted=(p[complete] >= np.array(thresholds)[None,:]).astype(int)
    correct=(actual == predicted).all(axis=1)
    positive_count=actual.sum(axis=1)
    output=[]
    for name, select in [('no_positive',positive_count==0),('single_positive',positive_count==1),
                         ('multiple_positive',positive_count>=2),('all_complete',np.ones(len(actual),dtype=bool))]:
        count=int(select.sum())
        output.append(dict(group=name,n_images=count,exact_correct=int(correct[select].sum()),
                           exact_accuracy=float(correct[select].mean()) if count else None))
    return output


def save_csv(path,rows):
    if not rows:
        return
    keys=list(rows[0].keys())
    with path.open('w',newline='',encoding='utf-8-sig') as f:
        writer=csv.DictWriter(f,fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)

## 3. 실행 설정
파일 경로를 확인하고 필요한 경우 수정하세요. 결과는 실행 시각별 폴더에 저장됩니다.

In [ ]:
from types import SimpleNamespace
args = SimpleNamespace(
    baseline=DEFAULT_BASELINE,
    weighted=DEFAULT_WEIGHTED,
    csv_dir=ROOT / 'data/processed',
    image_root=DEFAULT_IMAGE_ROOT,
    output_dir=None,
    batch_size=4,
)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Evaluation:',device,'| VALIDATION ONLY',flush=True)
out=args.output_dir or ROOT/'runs/validation_weighted_comparison'/datetime.now().strftime('%Y%m%d_%H%M%S')
out.mkdir(parents=True,exist_ok=True)

## 4. Validation 추론 및 모델 비교
기본 임계값 0.50에서 성능을 비교합니다. 이 셀에서 두 모델의 추론을 실행합니다.

In [ ]:
# The dataloader builder constructs metadata objects for all splits, but the
# script only iterates loaders['validate'] and never predicts Train/Test.
loaders=build_dataloaders(csv_dir=args.csv_dir,image_root=args.image_root,image_layout='flat',
                          image_size=224,batch_size=args.batch_size,num_workers=0,seed=42)
loader=loaders['validate']
results={}
metrics_rows=[]
comparison=[]
for model_name,checkpoint in [('baseline',args.baseline),('weighted',args.weighted)]:
    print('Evaluating',model_name,checkpoint,flush=True)
    model,meta=load_model(checkpoint,device)
    y,p=predict_validation(model,loader,device)
    del model
    if device.type=='cuda':
        torch.cuda.empty_cache()
    ranks=ranking_metrics(y,p)
    for k,finding in enumerate(TARGET_LABELS):
        v=confusion_and_metrics(y[:,k],p[:,k],0.5)
        v.update(model=model_name, **ranks[k])
        metrics_rows.append(v)
    averages=dict(model=model_name,macro_AUROC=float(np.nanmean([r['AUROC'] for r in ranks])),
                  macro_AP=float(np.nanmean([r['AP'] for r in ranks])),
                  mean_positive_F1=float(np.mean([metrics_rows[-3+i]['positive_f1'] for i in range(3)])))
    comparison.append(averages)
    results[model_name]=(y,p)
    print(model_name,averages,flush=True)
if not np.array_equal(results['baseline'][0],results['weighted'][0]):
    raise ValueError('두 모델의 Validation 정답 또는 데이터 순서가 서로 다릅니다!')

save_csv(out/'validation_model_comparison.csv',comparison)
save_csv(out/'validation_finding_metrics_threshold050.csv',metrics_rows)
display(pd.DataFrame(comparison))
display(pd.DataFrame(metrics_rows))

## 5. 소견별 임계값 탐색
0.10~0.90을 0.01 간격으로 탐색합니다. F1이 같으면 0.50에 가까운 값을 선택합니다.

In [ ]:
y,p=results['weighted']
sweep=[]
f1_best=[]
thresholds=np.round(np.arange(0.10,0.901,0.01),2)
for k,finding in enumerate(TARGET_LABELS):
    for threshold in thresholds:
        row=confusion_and_metrics(y[:,k],p[:,k],threshold)
        sweep.append(dict(finding=finding,**row))
    entries=[r for r in sweep if r['finding']==finding]
    max_f1=max(r['positive_f1'] for r in entries)
    tied=[r for r in entries if abs(r['positive_f1']-max_f1)<1e-12]
    best=min(tied,key=lambda r:abs(r['threshold']-0.5))
    f1_best.append(dict(best))
save_csv(out/'weighted_threshold_sweep_010_090.csv',sweep)
save_csv(out/'weighted_best_f1_thresholds.csv',f1_best)
display(pd.DataFrame(f1_best))

## 6. 심비대 임계값 후보 비교
민감도(recall), 특이도, 양성 F1을 함께 확인하세요.

In [ ]:
candidates=[]
for threshold in [0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60]:
    candidates.append(dict(finding='Cardiomegaly',**confusion_and_metrics(y[:,0],p[:,0],threshold)))
save_csv(out/'weighted_cardiomegaly_threshold_candidates.csv',candidates)
display(pd.DataFrame(candidates))

## 7. 단일·복합 소견 그룹 비교
폐부종 0.40, 흉수 0.50을 고정하고 심비대 후보를 비교합니다.

In [ ]:
# Group-level exact-match analysis; other thresholds 0.4/0.5 held fixed
# for illustrative cardiomegaly candidates (NOT automatically selected).
groups=[]
for tc in [0.40,0.45,0.50]:
    for item in group_accuracy(y,p,[tc,0.40,0.50]):
        groups.append(dict(cardiomegaly_threshold=tc,edema_threshold=0.4,
                           effusion_threshold=0.5,**item))
save_csv(out/'weighted_group_exactmatch_cardiomegaly_candidates.csv',groups)
display(pd.DataFrame(groups))

## 8. 예측 및 실행 정보 저장
Validation 행 순서에 따른 두 모델의 확률과 실행 정보를 저장합니다.

In [ ]:
# Save aligned per-image predictions for follow-up analyses; no patient ID
# is inferred. Row index refers to Validation dataset iteration order.
predictions=[]
base_p=results['baseline'][1]
for i in range(len(y)):
    row={'validation_row':i}
    for k,short in enumerate(('cardiomegaly','edema','pleural_effusion')):
        row[f'{short}_truth']=int(y[i,k])
        row[f'{short}_baseline_prob']=float(base_p[i,k])
        row[f'{short}_weighted_prob']=float(p[i,k])
    predictions.append(row)
save_csv(out/'validation_predictions_both_models.csv',predictions)

metadata={'dataset':'MIMIC-CXR Validation only','n_images':int(len(y)),
          'baseline_checkpoint':str(args.baseline),'weighted_checkpoint':str(args.weighted),
          'threshold_sweep':'0.10..0.90 step 0.01',
          'comparison_threshold':0.5,
          'note':'No Test predictions; best F1 thresholds are candidates, not automatically selected.'}
(out/'evaluation_info.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
print('\n=== Model comparison (threshold=0.50 for F1) ===')
for item in comparison:
    print(item)
print('\n=== Weighted F1-optimal threshold candidates (Validation) ===')
for item in f1_best:
    print(item['finding'],item['threshold'],'F1',round(item['positive_f1'],4),
          'Recall',round(item['recall'],4),'Specificity',round(item['specificity'],4))
print('\nSaved:',out,flush=True)

## 결과 확인

- `weighted_best_f1_thresholds.csv`: 소견별 F1 최대 임계값 후보
- `weighted_threshold_sweep_010_090.csv`: 전체 임계값별 지표
- `weighted_cardiomegaly_threshold_candidates.csv`: 심비대 후보 비교
- `weighted_group_exactmatch_cardiomegaly_candidates.csv`: 그룹별 세 소견 모두 정답 비율

후보를 검토한 뒤 최종 임계값을 추론 코드에 별도로 반영하세요. 임계값 변경만으로 재학습할 필요는 없습니다.